In [1]:
import pandas as pd
from pandas.api.types import is_numeric_dtype
import numpy as np
import torch
import torch.nn as nn
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from gplearn.genetic import SymbolicTransformer
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv(r'playground-series-s6e3\\train.csv')
if 'id' in df.columns:
    df = df.drop(columns=['id'])

target_col = 'Churn'
if target_col in df.columns and not is_numeric_dtype(df[target_col]):
    df[target_col] = df[target_col].map({'Yes':1,'No':0,'Y':1,'N':0,'True':1,'False':0})
    df[target_col] = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

C:\Users\Saswat Balyan\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_bins(df, cols, n_bins=5):
    df_out = df.copy()
    for c in cols:
            s = df_out[c]
            if is_numeric_dtype(s):
                try:
                    df_out[f'{c}_qcut'] = pd.qcut(s, q=n_bins, labels=False, duplicates='drop')
                except Exception:
                    df_out[f'{c}_qcut'] = pd.qcut(pd.to_numeric(s, errors='coerce'), q=n_bins, labels=False, duplicates='drop')
                try:
                    df_out[f'{c}_cut'] = pd.cut(s, bins=n_bins, labels=False)
                except Exception:
                    df_out[f'{c}_cut'] = pd.cut(pd.to_numeric(s, errors='coerce'), bins=n_bins, labels=False)
            else:
                
                df_out[f'{c}_qcut'] = s.astype('category').cat.codes
                df_out[f'{c}_cut'] = s.astype('category').cat.codes
    return df_out

def extract_digits(df, cols):
    df_out = df.copy()
    for c in cols:
        s = df_out[c]
        if is_numeric_dtype(s):
            num = pd.to_numeric(s, errors='coerce').fillna(0)
        else:
            num = s.astype('category').cat.codes

        df_out[f'{c}_unit'] = (np.floor(num) % 10).astype(int)
        df_out[f'{c}_tens'] = (np.floor(num / 10) % 10).astype(int)
        df_out[f'{c}_dec'] = (np.floor(num * 10) % 10).astype(int)
    return df_out

In [3]:
def to_categorical(df, cols):
    df_out = df.copy()
    for c in cols:
        df_out[f'{c}_str'] = df[c].astype(str).astype('category')
    return df_out

def freq_encoding(df, cols):
    df_out = df.copy()
    for c in cols:
        freq = df[c].value_counts()
        df_out[f'{c}_freq'] = df[c].map(freq)
    return df_out

In [4]:
class DVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, latent_dim * 2))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, input_dim))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        x_noisy = x + 0.1 * torch.randn_like(x)
        h = self.encoder(x_noisy)
        mu, logvar = h.chunk(2, dim=-1)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def add_gp_features(X, y):
    gp = SymbolicTransformer(generations=3, population_size=100, random_state=42)
    gp.fit(X, y)
    return gp.transform(X)

In [5]:
def train_cv_model(model_fn, X, y, fit_params=None):
    if fit_params is None: fit_params = {}
    oof = np.zeros(len(y))
    best_iters = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        model = model_fn()
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], **fit_params)
        
        oof[val_idx] = model.predict_proba(X_va)[:, 1]
        
        if hasattr(model, 'best_iteration'): best_iters.append(model.best_iteration)
        elif hasattr(model, 'best_iteration_'): best_iters.append(model.best_iteration_)
        elif hasattr(model, 'get_best_iteration'): best_iters.append(model.get_best_iteration())
            
    auc = roc_auc_score(y, oof)
    avg_iter = int(np.mean(best_iters)) if best_iters else None
    return oof, auc, avg_iter

In [6]:
features = [c for c in df.columns if c != target_col]
df_binned = create_bins(df, features)
df_digits = extract_digits(df_binned, features)

datasets = {
    'base': df[features],
    'binned': df_binned.drop(columns=[target_col], errors='ignore'),
    'digits': df_digits.drop(columns=[target_col], errors='ignore')
}
y = df[target_col]

for name, ds in datasets.items():
    ds = ds.copy()
    obj_cols = ds.select_dtypes(include=['object', 'string']).columns.tolist()
    for c in obj_cols:
        ds[c] = ds[c].astype('category').cat.codes
    datasets[name] = ds

oofs = {}
best_iterations = {}

for name, X_data in datasets.items():
    oofs[f'xgb_{name}'], _, best_iterations[f'xgb_{name}'] = train_cv_model(
        lambda: xgb.XGBClassifier(n_estimators=1000, eval_metric='auc', early_stopping_rounds=50), 
        X_data, y, {'verbose': False}
    )
    
    oofs[f'lgb_{name}'], _, best_iterations[f'lgb_{name}'] = train_cv_model(
        lambda: lgb.LGBMClassifier(n_estimators=1000), 
        X_data, y, {'callbacks': [lgb.early_stopping(50, verbose=False)]}
    )
    
    oofs[f'cb_{name}'], _, best_iterations[f'cb_{name}'] = train_cv_model(
        lambda: cb.CatBoostClassifier(iterations=1000, eval_metric='AUC', early_stopping_rounds=50), 
        X_data, y, {'verbose': False}
    )

df_oofs = pd.DataFrame(oofs)

[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010902 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 626
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011005 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 626
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 19
[LightGBM] [In

In [7]:
def objective(trial):
    selected_cols = [c for c in df_oofs.columns if trial.suggest_categorical(f'use_{c}', [True, False])]
    if not selected_cols: return 0.0
        
    X_blend = df_oofs[selected_cols]
    blend_oof = np.zeros(len(y))
    
    for train_idx, val_idx in skf.split(X_blend, y):
        X_tr, y_tr = X_blend.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_blend.iloc[val_idx], y.iloc[val_idx]
        
        model = Ridge()
        model.fit(X_tr, y_tr)
        blend_oof[val_idx] = model.predict(X_va)
        
    return roc_auc_score(y, blend_oof)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=False)

best_cols = [c for c in df_oofs.columns if study.best_trial.params.get(f'use_{c}')]
print(f"Selected {len(best_cols)} models out of {len(df_oofs.columns)}")

[I 2026-03-04 23:48:33,889] A new study created in memory with name: no-name-7c4ffb20-6c43-4744-a13b-cdfa6662d015
[I 2026-03-04 23:48:34,328] Trial 0 finished with value: 0.9173590832005064 and parameters: {'use_xgb_base': False, 'use_lgb_base': False, 'use_cb_base': True, 'use_xgb_binned': False, 'use_lgb_binned': True, 'use_cb_binned': False, 'use_xgb_digits': True, 'use_lgb_digits': True, 'use_cb_digits': False}. Best is trial 0 with value: 0.9173590832005064.
[I 2026-03-04 23:48:34,770] Trial 1 finished with value: 0.9172294243666165 and parameters: {'use_xgb_base': True, 'use_lgb_base': True, 'use_cb_base': True, 'use_xgb_binned': True, 'use_lgb_binned': False, 'use_cb_binned': False, 'use_xgb_digits': True, 'use_lgb_digits': False, 'use_cb_digits': False}. Best is trial 0 with value: 0.9173590832005064.
[I 2026-03-04 23:48:35,250] Trial 2 finished with value: 0.9176249152702525 and parameters: {'use_xgb_base': True, 'use_lgb_base': False, 'use_cb_base': True, 'use_xgb_binned': Tr

Selected 5 models out of 9


In [8]:
X_final_blend = df_oofs[best_cols]
final_ridge = Ridge()
final_ridge.fit(X_final_blend, y)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [9]:
full_models = []

for name in best_cols:
    model_type, feat_set = name.split('_')
    target_iter = int(best_iterations[name] * 1.25) if best_iterations[name] else 500
    X_train = datasets[feat_set]
    
    for seed in range(20):
        if model_type == 'xgb':
            m = xgb.XGBClassifier(n_estimators=target_iter, random_state=seed)
        elif model_type == 'lgb':
            m = lgb.LGBMClassifier(n_estimators=target_iter, random_state=seed)
        else:
            m = cb.CatBoostClassifier(iterations=target_iter, random_seed=seed, verbose=False)
            
        m.fit(X_train, y)
        full_models.append((m, feat_set))

[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023353 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 861
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 80
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235573
[LightGBM] [Info] Start training from score -1.235573
[LightGBM] [Info] Number of positive: 133817, number of negative: 460377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026512 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 861
[LightGBM] [Info] Number of data points in the train set: 594194, number of used features: 80
[LightGBM] [In

In [10]:
test_df = pd.read_csv(r'playground-series-s6e3\test.csv')
test_ids = test_df['id']
test_features = [c for c in test_df.columns if c != 'id']

test_binned = create_bins(test_df, test_features)
test_digits = extract_digits(test_binned, test_features)

test_datasets = {
    'base': test_df[test_features],
    'binned': test_binned.drop(columns=['id'], errors='ignore'),
    'digits': test_digits.drop(columns=['id'], errors='ignore')
}

for name, ds in test_datasets.items():
    ds = ds.copy()
    obj_cols = ds.select_dtypes(include=['object', 'string']).columns.tolist()
    for c in obj_cols:
        ds[c] = ds[c].astype('category').cat.codes
    test_datasets[name] = ds

test_preds = {}
model_idx = 0

for name in best_cols:
    model_type, feat_set = name.split('_', 1) 
    X_test = test_datasets[feat_set]
    
    avg_preds = np.zeros(len(test_df))
    for seed in range(20):
        m, _ = full_models[model_idx]
        avg_preds += m.predict_proba(X_test)[:, 1]
        model_idx += 1
        
    test_preds[name] = avg_preds / 20.0

X_test_blend = pd.DataFrame(test_preds)[best_cols]

final_predictions = final_ridge.predict(X_test_blend)
final_predictions = np.clip(final_predictions, 0, 1) 

submission = pd.DataFrame({
    'id': test_ids,
    'Churn': final_predictions
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()

Submission saved to submission.csv


,id,Churn
0,594194,0.084774
1,594195,0.000632
2,594196,0.106397
3,594197,0.003569
4,594198,0.510407
